# Stage Two: Cleaning and Features

This notebook turns the raw lots into a like-for-like sample and adds the features used in the analysis:

- **when**: `Then` for the 2023 editions, `Now` for 2026
- **era**: a bucket based on model year
- **decade**, **price_m** and **log_price**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
THEN, NOW = "#7FA3BF", "#B8945A"

In [2]:
lots = pd.read_csv("../data/raw/top_lots.csv")
totals = pd.read_csv("../data/raw/sale_totals.csv")

## Stage Two - Part One: A like-for-like sample

In [3]:
monterey = (lots[lots["week"] == "Monterey"]
            .sort_values("price_usd", ascending=False)
            .groupby("period")
            .head(30))
others = lots[lots["week"] != "Monterey"]
sample = pd.concat([monterey, others], ignore_index=True)

sample.pivot_table(index="week", columns="period", values="car", aggfunc="count")

period,2023,2026
week,,
Amelia,10,10
Arizona (RM Sotheby's),12,12
Monterey,30,30
Paris (RM Sotheby's),15,15


Every sale now has the same number of cars in each edition: 67 lots then and 67 now.

## Stage Two - Part Two: Features

In [4]:
ERA_ORDER = ["Pre-war", "1946-69", "1970s", "1980s-90s", "2000s on"]

def era(year):
    if year <= 1945:
        return "Pre-war"
    if year <= 1969:
        return "1946-69"
    if year <= 1979:
        return "1970s"
    if year <= 1999:
        return "1980s-90s"
    return "2000s on"

sample["when"] = np.where(sample["period"] <= 2023, "Then", "Now")
sample["era"] = pd.Categorical(sample["year"].apply(era), categories=ERA_ORDER, ordered=True)
sample["decade"] = (sample["year"] // 10) * 10
sample["price_m"] = sample["price_usd"] / 1e6
sample["log_price"] = np.log10(sample["price_usd"])
sample["built_1980_on"] = sample["year"] >= 1980

sample[["week", "when", "year", "era", "decade", "car", "price_m"]].head(12)

,week,when,year,era,decade,car,price_m
0,Monterey,Now,1964,1946-69,1960,Shelby Cobra Daytona Coupe,42.905
1,Monterey,Now,1996,1980s-90s,1990,McLaren F1 GTR,34.655
2,Monterey,Then,1967,1946-69,1960,Ferrari 412P Berlinetta,30.255
3,Monterey,Now,1963,1946-69,1960,Chevrolet Corvette Grand Sport,18.705
4,Monterey,Now,1963,1946-69,1960,Ferrari 250 P,18.705
5,Monterey,Now,2023,2000s on,2020,Ferrari Daytona SP3,17.825
6,Monterey,Now,1996,1980s-90s,1990,Ferrari F50,14.575
7,Monterey,Then,1957,1946-69,1950,Jaguar XKSS,13.205
8,Monterey,Now,1995,1980s-90s,1990,Ferrari F50,12.105
9,Monterey,Now,2003,2000s on,2000,Ferrari Enzo,12.100


In [5]:
sample.groupby("when")[["year", "price_m"]].describe().round(1).T

when              Now    Then
year    count    67.0    67.0
        mean   1987.9  1970.3
        std      28.1    25.9
        min    1928.0  1912.0
        25%    1962.5  1957.5
        50%    1994.0  1967.0
        75%    2010.0  1990.0
        max    2025.0  2022.0
price_m count    67.0    67.0
        mean      7.6     4.0
        std       7.2     4.3
        min       0.9     0.9
        25%       3.6     2.2
        50%       5.5     3.3
        75%       9.2     4.1
        max      42.9    30.3

## Stage Two - Part Three: Sale totals

In [6]:
t = totals.copy()
t["when"] = t.groupby("week")["period"].rank().map({1.0: "Then", 2.0: "Now"})
wide = t.pivot(index="week", columns="when", values="total_usd_m")[["Then", "Now"]]
wide["change_pct"] = (wide["Now"] / wide["Then"] - 1) * 100
wide = wide.loc[["Arizona", "Paris", "Amelia", "Monterey", "London (RM)"]]
wide.round(1)

when,Then,Now,change_pct
week,,,
Arizona,265.5,269.2,1.4
Paris,127.1,190.0,49.5
Amelia,178.4,255.9,43.4
Monterey,400.1,755.6,88.9
London (RM),31.6,31.5,-0.3


In [7]:
sample.to_csv("../data/processed/top_lots_clean.csv", index=False)
wide.to_csv("../data/processed/sale_totals_clean.csv")
print("Saved", len(sample), "lots and", len(wide), "sales")

Saved 134 lots and 5 sales
